# 03 — Full DBSCAN Clustering + CIS Computation

**Prerequisite**: `02_cluster_tuning.ipynb` must be complete and params committed to `configs/model.yaml`.

**What happens here**:
1. Load committed eps + min_samples from `configs/model.yaml`
2. Run DBSCAN on full 268k-row dataset → assigns `zone_id` to every row
3. Compute Congestion Impact Score (CIS) per zone (formula from `configs/eval.yaml`)
4. Run Phase B aggregation → zone × hour and zone × day grids (regression targets)
5. Save all outputs for training

**Files saved**:
- `data/processed/features_with_zones.parquet` — full row-level data with zone_id
- `data/processed/cis_table.parquet` — CIS score per zone
- `data/processed/cluster_stats.json` — DBSCAN run stats
- `data/processed/zone_hour_grid.parquet` — zone × hour aggregation (primary target)
- `data/processed/zone_day_grid.parquet` — zone × day aggregation (fallback target)

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock R2`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\USER\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Load committed params from configs
**What this cell does**: Reads eps + min_samples from model.yaml (not hardcoded).

**Expected output**:
```
DBSCAN params from model.yaml:
  eps         : 0.05
  min_samples : 50
```

In [3]:
import yaml
import pandas as pd

MODEL_CFG_PATH = PROJECT_ROOT / 'configs' / 'model.yaml'
EVAL_CFG_PATH  = PROJECT_ROOT / 'configs' / 'eval.yaml'

with open(MODEL_CFG_PATH) as f:
    model_cfg = yaml.safe_load(f)
with open(EVAL_CFG_PATH) as f:
    eval_cfg = yaml.safe_load(f)

EPS         = model_cfg['dbscan']['eps']
MIN_SAMPLES = model_cfg['dbscan']['min_samples']
SEED        = model_cfg['seed']

print('DBSCAN params from model.yaml:')
print(f'  eps         : {EPS}')
print(f'  min_samples : {MIN_SAMPLES}')
print(f'  seed        : {SEED}')

DBSCAN params from model.yaml:
  eps         : 0.05
  min_samples : 50
  seed        : 42


## Cell 4 — Load feature-engineered data
**Expected output**: `Loaded: (268281, 22)`

In [4]:
PARQUET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_row_level.parquet'

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f'features_row_level.parquet not found. Run 01b_features.ipynb first.'
    )

df = pd.read_parquet(PARQUET_PATH)
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

Loaded: (268281, 22)
Columns: ['latitude', 'longitude', 'vehicle_type', 'violation_type', 'offence_code', 'created_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'violation_type_primary', 'is_at_junction', 'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded']


## Cell 5 — Run DBSCAN on full dataset
**What this cell does**: Assigns zone_id to every row using committed params.

**Runtime**: ~2–3 minutes on 268k rows.

**Expected output**:
- Progress bar (3 steps)
- `DBSCAN complete: ~90 clusters | ~11% noise → zone_id=-1`
- `silhouette: ~0.34`

In [5]:
from src.models.clustering import run_clustering, save_cluster_stats

df_zoned, cluster_stats = run_clustering(
    df,
    eps=EPS,
    min_samples=MIN_SAMPLES,
    random_state=SEED,
)

save_cluster_stats(cluster_stats, PROJECT_ROOT / 'data' / 'processed' / 'cluster_stats.json')

print(f'\nzone_id column added. Shape: {df_zoned.shape}')
print(f'n_clusters  : {cluster_stats["n_clusters"]}')
print(f'noise rows  : {cluster_stats["n_noise"]:,} ({cluster_stats["noise_pct"]}%)')
print(f'silhouette  : {cluster_stats["silhouette_score"]}')

21:24:16 | INFO     | Running DBSCAN on full dataset (268,281 rows) | eps=0.05, min_samples=50



ssigning zone_id: 100%|██████████| 3/3 [00:38<00:00, 12.80s/step]

21:25:00 | INFO     | ✓ DBSCAN complete: 139 clusters | 5,558 noise rows (2.07%) → zone_id=-1 (sparse zone) | silhouette=-0.0955
21:25:00 | INFO     | Cluster stats saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\cluster_stats.json'

zone_id column added. Shape: (268281, 23)
n_clusters  : 139
noise rows  : 5,558 (2.07%)
silhouette  : -0.0955


## Cell 6 — Sanity check: zone_id distribution
**Expected output**: zone_id=-1 (noise) count, then top 10 largest zones.

In [6]:
zone_counts = df_zoned['zone_id'].value_counts().sort_values(ascending=False)

print(f'Total unique zone_ids: {df_zoned["zone_id"].nunique()}')
print(f'Noise zone (zone_id=-1): {(df_zoned["zone_id"] == -1).sum():,} rows')
print(f'\nTop 10 zones by row count:')
print(zone_counts.head(10))

Total unique zone_ids: 140
Noise zone (zone_id=-1): 5,558 rows

Top 10 zones by row count:
zone_id
 2     162766
 7      17855
 4       5738
-1       5558
 3       5292
 36      4928
 14      4641
 27      4475
 10      4400
 39      4300
Name: count, dtype: int64


## Cell 7 — Save features_with_zones.parquet
**Expected output**: File path and size.

In [7]:
zones_path = PROJECT_ROOT / 'data' / 'processed' / 'features_with_zones.parquet'
df_zoned.to_parquet(zones_path, index=False)
print(f'Saved : {zones_path}')
print(f'Shape : {df_zoned.shape}')
print(f'Size  : {zones_path.stat().st_size / 1e6:.1f} MB')

Saved : C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\features_with_zones.parquet
Shape : (268281, 23)
Size  : 8.4 MB


## Cell 8 — Compute CIS per zone
**What this cell does**: Computes Congestion Impact Score for each zone using formula v1.0.

Formula: `CIS = violation_density_norm × junction_weight`
- Noise zone (zone_id=-1): `junction_weight` overridden to 0.5

**Expected output**: HIGH/MEDIUM/LOW zone counts.

In [8]:
from src.models.clustering import compute_cis, save_cis_table

cis_df = compute_cis(df_zoned, eval_cfg)
save_cis_table(cis_df, PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet')

print(f'\nCIS table shape: {cis_df.shape}')
print(f'\nTop 10 zones by CIS score:')
print(cis_df.sort_values('cis_score', ascending=False).head(10).to_string(index=False))

21:25:01 | INFO     | Computing CIS v1.0 for 140 zones ...


Computing CIS scores:  75%|███████▌  | 3/4 [00:00<00:00, 92.49step/s]       

21:25:01 | INFO     |   Noise zone (zone_id=-1): cis_weight_override=0.5 applied


Computing CIS scores: 100%|██████████| 4/4 [00:00<00:00, 113.76step/s]

21:25:01 | INFO     | ✓ CIS complete: HIGH=1 zones | MEDIUM=0 zones | LOW=139 zones | cis_score_norm range [0.0002, 1.0000]
21:25:01 | INFO     | CIS table saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\cis_table.parquet'

CIS table shape: (140, 9)

Top 10 zones by CIS score:
 zone_id  violation_count  violation_density_norm  has_junction  junction_weight  cis_score formula_version priority_tier  cis_score_norm
       2           162766                1.000000             1              1.5   1.500000             1.0          HIGH        1.000000
       7            17855                0.109697             0              1.0   0.109697             1.0           LOW        0.073131
      27             4475                0.027493             1              1.5   0.041239             1.0           LOW        0.027493
      10             4400                0.027033             1              1.5   0.040550             1.0           LOW        0.027033
      29     

## Cell 9 — Zone × Hour aggregation (primary target)
**What this cell does**: Runs Phase B aggregation at hour granularity → `zone_hour_violation_count`.
This is the PRIMARY regression target for the ML model.

**Expected output**: Shape `(N_zone_hour_combos, ~10 cols)`, mean count, max count.

In [9]:
from src.data.features import aggregate_to_zone_grid

hour_grid_path = PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet'

zone_hour_df = aggregate_to_zone_grid(
    df_zoned,
    time_resolution='hour',
    save_path=hour_grid_path,
)

print(f'Zone×Hour grid shape : {zone_hour_df.shape}')
print(f'Columns              : {list(zone_hour_df.columns)}')
print(f'Target col stats:')
print(zone_hour_df['zone_hour_violation_count'].describe())

21:25:01 | INFO     | Aggregating to zone × hour grid ...


Aggregating (zone×hour): 100%|██████████| 4/4 [00:23<00:00,  5.89s/step]

21:25:25 | INFO     | ✓ Aggregation complete (hour): 26,354 zone×time rows | 140 unique zones | target='zone_hour_violation_count' (mean=10.18, max=284) | rolling_7d_count non-zero: 24,498/26,354
21:25:25 | INFO     | Aggregated grid saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\zone_hour_grid.parquet'
Zone×Hour grid shape : (26354, 16)
Columns              : ['zone_id', 'date', 'hour_of_day', 'zone_hour_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'day_of_week', 'month', 'rolling_7d_count']
Target col stats:
count    26354.000000
mean        10.179897
std         24.209280
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max        284.000000
Name: zone_hour_violation_count, dtype: float64


## Cell 10 — Zone × Day aggregation (fallback target)
**What this cell does**: Runs Phase B at day granularity → `zone_day_violation_count`.
Trained alongside hour model — winner chosen by NDCG@10.

**Expected output**: Shape `(N_zone_day_combos, ~10 cols)`.

In [10]:
day_grid_path = PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet'

zone_day_df = aggregate_to_zone_grid(
    df_zoned,
    time_resolution='day',
    save_path=day_grid_path,
)

print(f'Zone×Day grid shape : {zone_day_df.shape}')
print(f'Target col stats:')
print(zone_day_df['zone_day_violation_count'].describe())

21:25:25 | INFO     | Aggregating to zone × day grid ...


Aggregating (zone×day): 100%|██████████| 4/4 [00:06<00:00,  1.75s/step]

21:25:32 | INFO     | ✓ Aggregation complete (day): 8,246 zone×time rows | 140 unique zones | target='zone_day_violation_count' (mean=32.53, max=1848) | rolling_7d_count non-zero: 8,106/8,246
21:25:32 | INFO     | Aggregated grid saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\zone_day_grid.parquet'
Zone×Day grid shape : (8246, 15)
Target col stats:
count    8246.000000
mean       32.534683
std       148.376908
min         1.000000
25%         1.000000
50%         4.000000
75%        15.000000
max      1848.000000
Name: zone_day_violation_count, dtype: float64


## Cell 11 — Sparsity check
**What this cell does**: Checks how many zone×hour combos have zero violations.
High sparsity (>70%) means per-day may be the better target.

**What to look for**: If `zone_hour_violation_count` has >80% zeros in the TRAINING set,
consider flagging this before training.

In [11]:
# Note: the grid only contains zone×time combos WHERE violations occurred.
# The true sparsity = 1 - (observed combos / total possible combos)
n_zones        = df_zoned['zone_id'].nunique()
n_hours        = 24
n_days_train   = 113   # Nov 9 2023 → Feb 29 2024

total_possible_hour = n_zones * n_hours * n_days_train
observed_hour       = len(zone_hour_df[
    pd.to_datetime(zone_hour_df['date']) <= pd.Timestamp('2024-02-29')
])
sparsity_pct = round((1 - observed_hour / total_possible_hour) * 100, 1)

print(f'Zones            : {n_zones}')
print(f'Total possible   : {total_possible_hour:,} zone×hour combos (train window)')
print(f'Observed non-zero: {observed_hour:,}')
print(f'Sparsity         : {sparsity_pct}%')
print()
if sparsity_pct > 85:
    print('⚠  HIGH SPARSITY — per-day model may outperform per-hour. Watch NDCG@10.')
else:
    print('✓  Sparsity acceptable. Per-hour model should be viable.')

Zones            : 140
Total possible   : 379,680 zone×hour combos (train window)
Observed non-zero: 19,870
Sparsity         : 94.8%

⚠  HIGH SPARSITY — per-day model may outperform per-hour. Watch NDCG@10.


## Summary

**What was done**:
- DBSCAN run on full 268k rows with committed params (eps=0.05, min_samples=50)
- zone_id assigned to every row (-1 = sparse/noise zone)
- CIS computed per zone (formula v1.0: density_norm × junction_weight)
- Zone × Hour grid produced (primary regression target)
- Zone × Day grid produced (fallback target)

**What was saved**:
- `data/processed/features_with_zones.parquet`
- `data/processed/cluster_stats.json`
- `data/processed/cis_table.parquet`
- `data/processed/zone_hour_grid.parquet`
- `data/processed/zone_day_grid.parquet`

**Next step**: `notebooks/04_training.ipynb` — train XGBoost + LightGBM + CatBoost

In [12]:
print('=== Clustering Summary ===')
print(f'  eps              : {cluster_stats["eps"]}')
print(f'  min_samples      : {cluster_stats["min_samples"]}')
print(f'  n_clusters       : {cluster_stats["n_clusters"]}')
print(f'  noise_pct        : {cluster_stats["noise_pct"]}%')
print(f'  silhouette       : {cluster_stats["silhouette_score"]}')
print(f'  zone×hour rows   : {len(zone_hour_df):,}')
print(f'  zone×day rows    : {len(zone_day_df):,}')
print(f'  CIS zones        : {len(cis_df)}')
print(f'  Saved: data/processed/features_with_zones.parquet')
print(f'  Saved: data/processed/cluster_stats.json')
print(f'  Saved: data/processed/cis_table.parquet')
print(f'  Saved: data/processed/zone_hour_grid.parquet')
print(f'  Saved: data/processed/zone_day_grid.parquet')
print(f'  Next notebook: notebooks/04_training.ipynb')

=== Clustering Summary ===
  eps              : 0.05
  min_samples      : 50
  n_clusters       : 139
  noise_pct        : 2.07%
  silhouette       : -0.0955
  zone×hour rows   : 26,354
  zone×day rows    : 8,246
  CIS zones        : 140
  Saved: data/processed/features_with_zones.parquet
  Saved: data/processed/cluster_stats.json
  Saved: data/processed/cis_table.parquet
  Saved: data/processed/zone_hour_grid.parquet
  Saved: data/processed/zone_day_grid.parquet
  Next notebook: notebooks/04_training.ipynb
